In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, train_test_split, StratifiedKFold, cross_val_predict, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler, OrdinalEncoder
from sklearn.metrics import roc_auc_score, precision_recall_curve, average_precision_score, f1_score, accuracy_score
from sklearn.isotonic import IsotonicRegression

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import optuna
from optuna.integration import XGBoostPruningCallback
from optuna.integration import LightGBMPruningCallback
from optuna.integration import CatBoostPruningCallback

# Import Loan Dataset

In [ ]:
df = pd.read_excel("Uncapped_Strategy_Analyst_Case_Data.xlsx", sheet_name="Data")

In [ ]:
# Check data type
df.info()

# Data Cleaning

In [ ]:
# Check for nulls
df.isna().sum()

In [ ]:
# Remove rows containing nulls
df = df.dropna(axis=0)

In [ ]:
# Check for nulls
df.isna().sum()

# Feature Engineering

Use SQL for feature engineering.

Perform feature engineering, including.
- Ratios
    - loan_to_revenue = Loan Request / (Monthly Revenue + 1)
    - loan_to_cashrunway = Loan Request / (Cash Runway + 1)
- Application_quarter = Extract the quarter from the application date -> This is categorical features
- Number of past loans that was accepted
- Number of past loans that was defaulted
- Proportion of past loans that was accepted
- Proportion of past loans that was defaulted


In [ ]:
import duckdb as ddb

In [ ]:
# In-memory connection to DuckDB
con = ddb.connect()

# Register datagrame to DuckDB
con.register("loan_df", df) # name the "df" as "loan_df" in DuckDB

In [ ]:
# Loan to revenue ratio
df_enriched = con.query(
    '''
    WITH cte AS (
        SELECT
            *,

            COALESCE (COUNT("Loan_Accepted") OVER (
                PARTITION BY "Organisation"
                ORDER BY "Application Month" ASC
                ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
            ), 0) AS request_count,
            

            COALESCE (SUM("Loan_Accepted") OVER (
                PARTITION BY "Organisation"
                ORDER BY "Application Month" ASC
                ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
            ), 0) AS accepted_count,

            COALESCE (SUM("Default") OVER (
                PARTITION BY "Organisation"
                ORDER BY "Application Month" ASC
                ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
            ), 0) AS defaulted_count,


        FROM loan_df    
    )
    
    SELECT
        *,
        "Loan Requested" / ("Monthly Revenue" + 1) AS loan_to_revenue,
        "Loan Requested" / ("Cash Runway" + 1) AS loan_to_cashrunway,
        EXTRACT(QUARTER FROM "Application Month") AS application_quarter,
        accepted_count / (request_count + 0.0001) AS acceptance_rate,
        defaulted_count / (accepted_count + + 0.0001) AS default_rate,    
        
    FROM cte
    '''
).df()

# Feature Selection

Select relevant features (i.e. features that do not cause data leakage -> features with values that come post-decision) and the target variables ("Loan Accepted" and "Default")

In [ ]:
# Select features that causes no data leakage
selected_features = [
    'Organisation',
    'Business Type',
    'Biz Age',
    'Cash Runway',
    'Monthly Revenue',
    'Revenue Trend',
    'Loan Requested',
    'loan_to_revenue',
    'loan_to_cashrunway',
    'application_quarter',
    'accepted_count',
    'defaulted_count',
    'acceptance_rate',
    'default_rate'
]

target_variables = ['Loan_Accepted', 'Default']

df_selected = df_enriched.loc[:, selected_features + target_variables]

# Feature Transformation

In [ ]:
df_transformed = df_selected.copy()

In [ ]:
# List out features to be transformed
not_transformed = ['Revenue Trend', 'Loan_Accepted', 'Default']

num_features = ['Biz Age',
                'Cash Runway',
                'Monthly Revenue',
                'Loan Requested',
                'loan_to_revenue',
                'loan_to_cashrunway',
                'accepted_count',
                'defaulted_count',
                'acceptance_rate',
                'default_rate']

cat_features = ['Organisation', 'Business Type', 'application_quarter']

In [ ]:
# Perform robust scaling on categorical features
robust_scaler = RobustScaler()
df_transformed[num_features] = robust_scaler.fit_transform(df_transformed[num_features])

In [ ]:
# Perform one-hot-encoding on categorical features
df_transformed = pd.get_dummies(df_transformed, columns=cat_features)

# Separate Features and Target Variables

In [ ]:
y_acc = df_transformed["Loan_Accepted"]
y_def = df_transformed["Default"]
X = df_transformed.drop(["Loan_Accepted", "Default"], axis=1)

# Prediction Model: Loan Acceptance Probability

In [ ]:
class ThresholdTunedClassifier():
    """
    A meta-estimator that:
      1) tunes hyperparameters via inner CV (RandomizedSearchCV, scoring=AP by default),
      2) produces OOF probabilities via outer CV,
      3) (optionally) learns a calibration mapping on OOF probs (isotonic or Platt),
      4) selects the F1-optimal decision threshold on (calibrated or raw) OOF probs,
      5) refits the best model on the full training set, and
      6) predicts with the frozen threshold.
    """

    def __init__(
        self,
        random_state=123,
        n_jobs=-1,
        verbose=0,
    ):
        self.random_state = random_state
        self.n_jobs = n_jobs
        self.verbose = verbose

        # Learned attributes after fit()/tuning
        self.best_estimator_ = None
        self.calibrator_ = None
        self.use_calibration_ = False
        self.threshold_ = 0.5

    # ------------ helpers ------------

    @staticmethod
    def _max_f1_threshold(y_true, y_proba):
        """
        Find decision threshold that maximizes F1.

        Note: precision_recall_curve returns precision/recall length (n_thr+1),
        thresholds length (n_thr). Align F1 to thresholds using f1[:-1].
        """
        prec, rec, thr = precision_recall_curve(y_true, y_proba)
        f1 = 2 * prec * rec / (prec + rec + 1e-12)

        if thr.size == 0:
            # degenerate case (e.g., constant scores); fall back to 0.5
            return 0.5, float(f1_score(y_true, (y_proba >= 0.5).astype(int)))

        idx = int(np.nanargmax(f1[:-1]))     # align with thresholds
        optimal_thr = float(thr[idx])
        optimal_f1  = float(f1[:-1][idx])
        return optimal_thr, optimal_f1

    def _fit_calibrator(self, y_proba, y_true):
        """
        Fit probability calibrator (isotonic or Platt).
        """
        if self.calibration_method == "isotonic":
            iso = IsotonicRegression(out_of_bounds="clip")
            iso.fit(y_proba, y_true)
            return iso
        elif self.calibration_method == "platt":
            lr = LogisticRegression(solver="lbfgs", max_iter=1000)
            lr.fit(y_proba.reshape(-1, 1), y_true)  # requires 2D
            return lr
        else:
            raise ValueError("calibration_method must be 'isotonic' or 'platt'")

    def _apply_calibrator(self, calibrator, y_proba):
        """
        Apply fitted calibrator to raw probabilities.
        """
        if calibrator is None:
            return y_proba
        if isinstance(calibrator, IsotonicRegression):
            return calibrator.transform(y_proba)
        elif isinstance(calibrator, LogisticRegression):
            return calibrator.predict_proba(y_proba.reshape(-1, 1))[:, 1]
        else:
            raise TypeError("Unknown calibrator type")

    # ------------ core API ------------

    def model_tuning(self, X, y, base_estimator, param_distributions, n_iter=1, n_folds=3, tuner_scoring="average_precision", **kwargs):
        """
        Inner-CV hyperparameter tuning. Sets self.best_estimator_ and returns it.
        """
        self.base_estimator = base_estimator
        self.param_distributions = param_distributions
        self.n_iter = n_iter
        self.tuner_scoring = tuner_scoring

        cv = StratifiedKFold(
            n_splits=n_folds, shuffle=True, random_state=self.random_state
        )

        tuner = RandomizedSearchCV(
            estimator=self.base_estimator,
            param_distributions=self.param_distributions,
            n_iter=self.n_iter,
            scoring=self.tuner_scoring,
            cv=cv,
            n_jobs=self.n_jobs,
            random_state=self.random_state,
            verbose=self.verbose,
            refit=True,
        )
        tuner.fit(X, y, **kwargs)
        self.best_estimator_ = tuner.best_estimator_
        return self.best_estimator_
    
    def enter_best_model_config(self, best_model):
        """
        Manually define the hyper-parameter configuration for the best model
        """
        self.best_estimator_ = best_model
        return self.best_estimator_
    
    def decision_thr_tuning(self, X, y, n_folds=3, **kwargs):
        """
        Build OOF probabilities (using OUTER CV), compute F1-opt threshold,
        and return (oof_proba, thr_raw, f1_raw).
        """
        cv = StratifiedKFold(
            n_splits=n_folds, shuffle=True, random_state=self.random_state
        )

        oof_proba = cross_val_predict(
            self.best_estimator_, X, y, cv=cv, method="predict_proba", n_jobs=self.n_jobs, fit_params=kwargs if kwargs else None
        )[:, 1]

        thr_raw, f1_raw = self._max_f1_threshold(y, oof_proba)
        self.threshold_ = thr_raw
        return oof_proba, thr_raw, f1_raw

    def calibrate_prob(self, y, oof_proba, thr_raw, f1_raw, calibration_method="isotonic"):
        """
        Calibration learned on OOF probs.
        """
        self.calibration_method = calibration_method

        calibrator = self._fit_calibrator(oof_proba, y)
        oof_proba_cal = self._apply_calibrator(calibrator, oof_proba)
        thr_cal, f1_cal = self._max_f1_threshold(y, oof_proba_cal)

        use_cal = f1_cal > f1_raw

        if use_cal:
            self.use_calibration_ = True
            self.calibrator_ = calibrator
            self.threshold_ = thr_cal
        else:
            self.use_calibration_ = False
            self.calibrator_ = None
            self.threshold_ = thr_raw

        return self.threshold_

    def model_training(self, X, y, **kwargs):
        if self.best_estimator_ is None:
            raise RuntimeError("Call model_tuning() before model_training().")
        self.best_estimator_.fit(X, y, **kwargs)

    def generate_oof_proba_preds(self, X, y, n_folds, **kwargs):
        # Compute OOF probabilities
        cv = StratifiedKFold(
            n_splits=n_folds, shuffle=True, random_state=self.random_state
        )
        y_proba_pred = cross_val_predict(
            self.best_estimator_, X, y, cv=cv, method="predict_proba", n_jobs=self.n_jobs, fit_params=kwargs if kwargs else None
        )[:, 1]
        
        return y_proba_pred
    
    def analyze_performance(self, X, y, n_folds, **kwargs):
        # Compute OOF probabilities
        y_proba_pred = self.generate_oof_proba_preds(X, y, n_folds, **kwargs)
        
        # Apply calibration (if enabled)
        if self.use_calibration_:
            y_proba_eval = self._apply_calibrator(self.calibrator_, y_proba_pred)
        else:
            y_proba_eval = y_proba_pred
        y_proba_eval = np.clip(y_proba_eval, 0.0, 1.0)
        
        # Metrics independent on Decision Threshold
        roc_auc = roc_auc_score(y, y_proba_pred)
        average_precision = average_precision_score(y, y_proba_pred)
        
        # Metrics dependent on Decision Threshold
        y_pred = (y_proba_eval >= self.threshold_).astype(int)
        acc = accuracy_score(y, y_pred)
        f1_pos = f1_score(y, y_pred, pos_label=1, average="binary")  # class 1
        f1_neg = f1_score(y, y_pred, pos_label=0, average="binary")  # class 0
        f1_macro = f1_score(y, y_pred, average="macro")
        
        results = {
            "roc_auc": float(roc_auc),
            "average_precision": float(average_precision),
            "accuracy": float(acc),
            "f1_class_1": float(f1_pos),
            "f1_class_0": float(f1_neg),
            "f1_macro": float(f1_macro),
        }
        self.last_cv_metrics_ = results
        
        return results
    
    def predict_proba(self, X):
        """
        Predict class probabilities; applies calibration if enabled.
        Returns shape (n_samples, 2) = [P(class 0), P(class 1)].
        """
        if self.best_estimator_ is None:
            raise RuntimeError("Call model_training() before predict_proba().")
        proba = self.best_estimator_.predict_proba(X)[:, 1]
        if self.use_calibration_:
            proba = self._apply_calibrator(self.calibrator_, proba)
        proba = np.clip(proba, 0.0, 1.0)
        return np.vstack([1 - proba, proba]).T

    def predict(self, X):
        """
        Thresholded class prediction using self.threshold_.
        """
        p = self.predict_proba(X)[:, 1]
        return (p >= self.threshold_).astype(int)

### XGBoost Model

In [ ]:
# Perform hyper-parameter tuning using Optunna
def xgb_objective(trial):
    # define model hyper-parameters and hyper-parameter search space
    scale_pos_weight = (y_acc == 0).sum() / (y_acc == 1).sum() # From XGBoost documentation: the weight should be the ratio between the negative class / postiive class, not the other way around
    params = {
        "objective": 'binary:logistic',
        "eval_metric": ["aucpr"], # This has to be defined, because this is the metric that will be used during pruning
        "random_state": 98464,
        "scale_pos_weight": scale_pos_weight,
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 3e-1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 300, 1000, step=100),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 32.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 1.0),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.7, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 1e+1, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1e+1, log=True)
    }
    
    model = XGBClassifier(**params)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=67)
    scores = []

    for train_idx, val_idx in cv.split(X, y_acc):
        pruning_cb = XGBoostPruningCallback(trial, "validation_0-aucpr") # Boosting-round based pruning
        model.fit(
            X.iloc[train_idx,:], y_acc.iloc[train_idx],
            eval_set=[(X.iloc[val_idx,:], y_acc.iloc[val_idx])],
            callbacks=[pruning_cb],          # pruning each boosting round
            verbose=False
        )
        proba = model.predict_proba(X.iloc[val_idx,:])[:, 1]
        scores.append(average_precision_score(y_acc.iloc[val_idx], proba))

    return float(np.mean(scores))

# Define the pruner
xgb1_pruner = optuna.pruners.SuccessiveHalvingPruner(
    min_resource=1,          # start checking at first fold
    reduction_factor=3,      # roughly keep top 1/3 at each rung
    min_early_stopping_rate=0
)

# Perform hyper-parameter tuning
xgb1_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=2025, n_startup_trials=20),
    pruner=xgb1_pruner
)
xgb1_study.optimize(xgb_objective, n_trials=50)

In [ ]:
print("Best AP:", xgb1_study.best_value)
print("Best params:", xgb1_study.best_params)

In [ ]:
# Define a new ThresholdTunedClassifier object
xgb1 = ThresholdTunedClassifier()

# Define the optimized model
scale_pos_weight = (y_acc == 0).sum() / (y_acc == 1).sum() # From XGBoost documentation: the weight should be the ratio between the negative class and postiive class, not the other way around
xgb1.enter_best_model_config(XGBClassifier(
    objective = 'binary:logistic',
    eval_metric = ["aucpr"], # This has to be defined, because this is the metric that will be used during pruning
    random_state = 98464,
    scale_pos_weight = scale_pos_weight,
    **xgb1_study.best_params
))

In [ ]:
# Tune Decision Threshold
xgb1_oof_proba, xgb1_thr_raw, xgb1_f1_raw = xgb1.decision_thr_tuning(X, y_acc, n_folds=5)

In [ ]:
# Calibrate the model
xgb1_thr_cal = xgb1.calibrate_prob(y_acc, xgb1_oof_proba, xgb1_thr_raw, xgb1_f1_raw, calibration_method='isotonic')

In [ ]:
# Train the optimal model with the best set of hyper-parameters
xgb1.model_training(X, y_acc)

In [ ]:
# Analyze model performance
xgb1.analyze_performance(X, y_acc, n_folds=5)

Note that the model does not perform well. In practice, I would not trust this model to predict probability of loan acceptance. This is be caused by 3 things:
- Lack of predictive features -> I would try to enrich my dataset to improve model performance.
- The dataset is too small.
- The current underlying regime is inconsistent / questionable.

In [ ]:
# Generate probability prediction
xgb1_proba_pred = xgb1.predict_proba(X)[:,1]

# Prediction Model: Loan Default Probability

In [ ]:
# Generate weights(to be used later for the Loan Default model
xgb1_clipped_prob = xgb1_proba_pred.clip(0.05, 1) # Clip extremely small values because -> w = 1/prob
loan_default_weights = 1/xgb1_clipped_prob

In [ ]:
# Filter the dataset to only include data points where laon was approved
X_acc = X.loc[y_acc==1,: ]
X_acc.reset_index(drop=True, inplace=True)

y_def_filtered = y_def.loc[y_acc==1]
y_def_filtered.reset_index(drop=True, inplace=True)

loan_default_weights_filtered = loan_default_weights[y_acc==1]

### XGBoost Model

In [ ]:
# Perform hyper-parameter tuning using Optunna
def xgb2_objective(trial):
    # define model hyper-parameters and hyper-parameter search space
    scale_pos_weight = (y_def_filtered == 0).sum() / (y_def_filtered == 1).sum() # From XGBoost documentation: the weight should be the ratio between the negative class / postiive class, not the other way around
    params = {
        "objective": 'binary:logistic',
        "eval_metric": ["aucpr"], # This has to be defined, because this is the metric that will be used during pruning
        "random_state": 98464,
        "scale_pos_weight": scale_pos_weight,
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 3e-1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 300, 1000, step=100),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 32.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 1.0),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.7, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 1e+1, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1e+1, log=True)
    }
    
    model = XGBClassifier(**params)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=67)
    scores = []

    for train_idx, val_idx in cv.split(X_acc, y_def_filtered):
        pruning_cb = XGBoostPruningCallback(trial, "validation_0-aucpr") # Boosting-round based pruning
        model.fit(
            X_acc.iloc[train_idx,:], y_def_filtered.iloc[train_idx],
            eval_set=[(X_acc.iloc[val_idx,:], y_def_filtered.iloc[val_idx])],
            callbacks=[pruning_cb],          # pruning each boosting round
            verbose=False,
            sample_weight=loan_default_weights_filtered[train_idx] # Specify sample weight -> based on prob of loan acceptance
        )
        proba = model.predict_proba(X_acc.iloc[val_idx,:])[:, 1]
        scores.append(average_precision_score(y_def_filtered.iloc[val_idx], proba))

    return float(np.mean(scores))

# Define the pruner
xgb2_pruner = optuna.pruners.SuccessiveHalvingPruner(
    min_resource=1,          # start checking at first fold
    reduction_factor=3,      # roughly keep top 1/3 at each rung
    min_early_stopping_rate=0
)

# Perform hyper-parameter tuning
xgb2_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=2025, n_startup_trials=20),
    pruner=xgb2_pruner
)
xgb2_study.optimize(xgb2_objective, n_trials=50)

In [ ]:
print("Best AP:", xgb2_study.best_value)
print("Best params:", xgb2_study.best_params)

In [ ]:
# Define a new ThresholdTunedClassifier object
xgb2 = ThresholdTunedClassifier()

# Define the optimized model
scale_pos_weight = (y_def_filtered == 0).sum() / (y_def_filtered == 1).sum() # From XGBoost documentation: the weight should be the ratio between the negative class and postiive class, not the other way around
xgb2.enter_best_model_config(XGBClassifier(
    objective = 'binary:logistic',
    eval_metric = ["aucpr"], # This has to be defined, because this is the metric that will be used during pruning
    random_state = 98464,
    scale_pos_weight = scale_pos_weight,
    **xgb2_study.best_params
))

In [ ]:
# Tune Decision Threshold
xgb2_oof_proba, xgb2_thr_raw, xgb2_f1_raw = xgb2.decision_thr_tuning(X_acc, y_def_filtered, n_folds=5, sample_weight=loan_default_weights_filtered)

In [ ]:
# Calibrate the model
xgb2_thr_cal = xgb2.calibrate_prob(y_def_filtered, xgb2_oof_proba, xgb2_thr_raw, xgb2_f1_raw, calibration_method='isotonic')

In [ ]:
# Train the optimal model with the best set of hyper-parameters
xgb2.model_training(X_acc, y_def_filtered, sample_weight=loan_default_weights_filtered)

In [ ]:
# Analyze model performance
xgb2.analyze_performance(X_acc, y_def_filtered, n_folds=5, sample_weight=loan_default_weights_filtered)

In [ ]:
# Generate probability prediction on the complete dataset
xgb2_proba_pred = xgb2.predict_proba(X)[:,1]

# Segment Analysis

In [ ]:
# Add the default probability tour dataset
df_complete = df_enriched.copy()
df_complete["default_prob"] = xgb2_proba_pred

In [ ]:
df_complete.sort_values("default_prob", ascending=True, inplace=True)
df_complete.reset_index(drop=True, inplace=True)

Define fixed thresholds based on the probability of default,
- A (Low risk): default_prob < 10%
- B (Moderate risk): 10% <= default_prob < 20%
- C (High risk): 20% <= default_prob < 50%
- D (Very high risk): default_prob >= 50%

In [ ]:
# Add classification as define above
def assign_band(p):
    if p < 0.10:
        return "A"
    elif p < 0.20:
        return "B"
    elif p < 0.50:
        return "C"
    else:
        return "D"

df_complete["default_class"] = df_complete["default_prob"].apply(assign_band)

In [ ]:
# Export the completed dataset
df_complete.to_csv('df_complete.csv', index=False)

The summary report of each Band containing:
- Business profile and loan request characteristics
    - The % proportion of each business type (ecommerce vs saas vs other)
    - Average business age
    - Average Cash Runway
    - Average Monthly Revenue
    - The % proportion of each Revenue Trend (-1 vs 0 vs +1)
    - Average size of Loan Requested
    - Loan requested - to - revenue ratio
    - Loan requested - to - cashrunway ratio
    - Band-wide (loan_issue + fee_charged)-to-term ratio -> This represents the montly repayment, in which we can compare with the monthly revenue
    - Band-wide monthly-repayment-to-revenue ratio

- Underwriting behaviour -> Analyzing the current state of underwriting
    - Count of loan requests
    - Count of accepted loan requests
    - Proportions of accepted loans -> This is to analyze the current state of the underwriting process, e.g. whether loan requests categorized in Band A are generally accepted or not
    - Band-wide Fee Charged to Loan Issued ratio (only include accepted loans in the average) -> This allows us to see, for example, fees charged on Band A loan should be lower than fees charged on Band B loan
    
- Risk & return performance
    - Count of default loan
    - Default rate (only include accepted loans in the average) -> We can compare the accuracy of our model by checking whether the actual realized default rate in each band fits the band range
    - Band-wide loss-to-loan_issued ratio-> This allows us to see, for example, the ratio on Band A loan should be lower than ratio on Band B loan 
    - Band-wide % return
    - Band-wise total $ return

In [ ]:
# Register datagrame to DuckDB
con.register("complete_loan_df", df_complete)

# Summarize the characteristics of each class
summary_report = con.query(
    """
    WITH base AS (
        SELECT
            *,
            -- Ratios: loan requested to revenue / runway
            CASE 
                WHEN "Monthly Revenue" IS NOT NULL AND "Monthly Revenue" != 0
                THEN "Loan Requested" / "Monthly Revenue"
            END AS loan_to_revenue,
            CASE 
                WHEN "Cash Runway" IS NOT NULL AND "Cash Runway" != 0
                THEN "Loan Requested" / "Cash Runway"
            END AS loan_to_cashrunway
        FROM complete_loan_df
    ),
    high_quality_clients AS (
        -- Clients with >= 2 accepted, non-defaulting loans
        SELECT
            "Organisation",
            SUM(
                CASE 
                    WHEN "Loan_Accepted" = 1 AND "Default" = 0 THEN 1 
                    ELSE 0 
                END
            ) AS n_good_loans
        FROM base
        GROUP BY "Organisation"
    )
    
    SELECT
        b."default_class" AS band,



        -- Business profile & loan request characteristics
        AVG(CASE WHEN b."Business Type" = 'ecommerce' THEN 1.0 ELSE 0.0 END) AS prop_ecommerce,
        AVG(CASE WHEN b."Business Type" = 'saas' THEN 1.0 ELSE 0.0 END) AS prop_saas,
        AVG(CASE WHEN b."Business Type" = 'other' THEN 1.0 ELSE 0.0 END) AS prop_other,

        AVG(b."Biz Age") AS avg_business_age,
        AVG(b."Cash Runway") AS avg_cash_runway,
        AVG(b."Monthly Revenue") AS avg_monthly_revenue,

        AVG(CASE WHEN b."Revenue Trend" = -1 THEN 1.0 ELSE 0.0 END) AS prop_trend_down,
        AVG(CASE WHEN b."Revenue Trend" =  0 THEN 1.0 ELSE 0.0 END) AS prop_trend_flat,
        AVG(CASE WHEN b."Revenue Trend" =  1 THEN 1.0 ELSE 0.0 END) AS prop_trend_up,

        AVG(b."Loan Requested") AS avg_loan_request,
        AVG(loan_to_revenue) AS avg_loan_to_revenue,
        AVG(loan_to_cashrunway) AS avg_loan_to_cashrunway,

        -- Band-wide average monthly repayment: (Loan Issued + Fee) / Term on accepted loans
        AVG(
            CASE 
                WHEN b."Loan_Accepted" = 1 AND b."Term" != 0
                THEN (b."Loan Issued" + b."Fee Charged") / b."Term"
                ELSE NULL
            END
        ) AS avg_monthly_repayment__,
        
        -- Band-wide monthly repayment - to - monthly revenue ratio
        AVG(
            CASE 
                WHEN b."Loan_Accepted" = 1 AND b."Term" != 0 AND b."Monthly Revenue" != 0
                THEN ((b."Loan Issued" + b."Fee Charged") / b."Term") / b."Monthly Revenue"
                ELSE NULL
            END
        ) AS avg_repayment_to_revenue_ratio__,
        
        

        -- Underwriting behaviour
        COUNT(*) AS n_applications,
        SUM(b."Loan_Accepted") AS n_accepted,
        AVG(b."Loan_Accepted") AS acceptance_rate,
        SUM(b."Fee Charged") / NULLIF(SUM(b."Loan Issued"), 0) AS fee_to_loan_issued__,



        -- Risk & return performance (accepted loans only where needed)
        SUM(
            CASE 
                WHEN b."Loan_Accepted" = 1 AND b."Default" = 1 THEN 1 
                ELSE 0 
            END
        ) AS n_defaults,

        AVG(
            CASE 
                WHEN b."Loan_Accepted" = 1 THEN b."Default"
                ELSE NULL
            END
        ) AS default_rate_accepted__,
        
        -- Loss-to-loan_issued ratio (on accepted loans)
        SUM(
            CASE
                WHEN b."Loan_Accepted" = 1 THEN b."Loss (Default)"
                ELSE 0
            END
        ) 
        / NULLIF(
            SUM(
                CASE 
                    WHEN b."Loan_Accepted" = 1 THEN b."Loan Issued"
                    ELSE 0
                END
            ),
            0
        ) AS loss_to_loan_ratio__,

        -- Band-wide £ return: sum(Return £)
        SUM(
            CASE 
                WHEN b."Loan_Accepted" = 1 THEN
                    b."Fee Charged" - b."Loss (Default)" - b."Cost of Fund (lifetime)" - b."Cost of Operations (Lifetime)"
                ELSE 0
            END
        ) AS total_dollar_return__,        
        
        -- Band-wide % return: sum(Return £) / sum(Loan Issued) on accepted loans
        SUM(
            CASE 
                WHEN b."Loan_Accepted" = 1 THEN
                    b."Fee Charged" - b."Loss (Default)" - b."Cost of Fund (lifetime)" - b."Cost of Operations (Lifetime)"
                ELSE 0
            END
        )
        / NULLIF(
            SUM(
                CASE 
                    WHEN b."Loan_Accepted" = 1 THEN b."Loan Issued"
                    ELSE 0
                END
            ),
            0
        ) AS band_return_pct__,

        
    FROM base b
    LEFT JOIN high_quality_clients hq
        ON b."Organisation" = hq."Organisation"
    GROUP BY b."default_class"
    ORDER BY b."default_class"
    """
).df()

In [ ]:
summary_report.T

In [ ]:
# Export as CSV file
summary_report.T.to_csv("summary_report.csv")

# Recommendations

In [ ]:
# Copmtuing current, realized the global % Returns
df_complete["dollar_returns"] = df_complete["Fee Charged"] - df_complete["Loss (Default)"] - df_complete["Cost of Fund (lifetime)"] - df_complete["Cost of Operations (Lifetime)"]
global_pct_returns = df_complete["dollar_returns"].sum() / df_complete["Loan Issued"].sum() * 100
print(f'{global_pct_returns}%')

In [ ]:
# Compute the average predicted probabiliyt of default in each band
return_report = con.query(
    """
    WITH cte AS (
        SELECT
            "default_class",
            -- Loss-to-loan_issued ratio (on accepted loans)
            COALESCE(
                SUM(
                    CASE
                        WHEN "Default" = 1 THEN "Loss (Default)"
                        ELSE 0
                    END
                )
                / SUM(
                        CASE 
                            WHEN "Default" = 1 THEN "Loan Issued"
                            ELSE 0
                        END
                    )
            , 0) AS loss_to_loan_ratio__
        FROM complete_loan_df
        GROUP BY "default_class"
    )
    
    SELECT
        df."default_class",
        SUM(
            "Fee Charged"
            - (df."default_prob" * "Loan Requested" * cte_df.loss_to_loan_ratio__)
            - "Cost of Fund (lifetime)"
            - "Cost of Operations (Lifetime)"
        ) AS exp_dollar_returns,
        SUM(
            "Fee Charged"
            - (df."default_prob" * "Loan Requested" * cte_df.loss_to_loan_ratio__)
            - "Cost of Fund (lifetime)"
            - "Cost of Operations (Lifetime)"
        )
        / NULLIF(SUM("Loan Issued"), 0) * 100 AS exp_pct_returns
    FROM complete_loan_df AS df
    LEFT JOIN cte AS cte_df
    ON df."default_class" = cte_df."default_class"
    GROUP BY df."default_class"
    ORDER BY df."default_class"
    """
).df()

In [ ]:
return_report